In [2]:
import sys
from pathlib import Path

In [3]:
src_path = Path("../src").resolve()
sys.path.append(str(src_path))

In [ ]:
from sqlmodel import Session, select
from api.db.session import engine
from api.events.models import EventModel
from pprint import pprint
from timescaledb.hyperfunctions import time_bucket

In [9]:
with Session(engine) as session:
  query = select(EventModel).order_by(EventModel.update_at.asc()).limit(10)
  compiled_query = query.compile(compile_kwargs={"literal_binds": True})
  print(compiled_query)
  print("")
  print(str(query))
  results = session.exec(query).fetchall()
  pprint(results)

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.update_at 
FROM eventmodel ORDER BY eventmodel.update_at ASC
 LIMIT 10

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.update_at 
FROM eventmodel ORDER BY eventmodel.update_at ASC
 LIMIT :param_1
[EventModel(id=1, time=datetime.datetime(2026, 4, 18, 9, 25, 55, 821116, tzinfo=datetime.timezone.utc), description='', update_at=datetime.datetime(2026, 4, 18, 9, 25, 55, 821139, tzinfo=datetime.timezone.utc), page='pricing'),
 EventModel(id=2, time=datetime.datetime(2026, 4, 18, 9, 25, 55, 952429, tzinfo=datetime.timezone.utc), description='', update_at=datetime.datetime(2026, 4, 18, 9, 25, 55, 952444, tzinfo=datetime.timezone.utc), page='/contact'),
 EventModel(id=3, time=datetime.datetime(2026, 4, 18, 9, 25, 56, 1157, tzinfo=datetime.timezone.utc), description='', update_at=datetime.datetime(2026, 4, 18, 9, 25, 56, 1173, tzinfo=datetime.timezone.utc), page=

In [18]:
from sqlalchemy import func
from datetime import datetime, timedelta, timezone

with Session(engine) as session:
  bucket = time_bucket("1 min", EventModel.time)
  pages = ['/about', '/contact', '/pages', '/pricing']
  start = datetime.now(timezone.utc) - timedelta(hours=1)
  finish = datetime.now(timezone.utc) + timedelta(hours=1)
  query = (
      select(
        bucket,
        EventModel.page,
        func.count()
      )
      .where(
        EventModel.time > start,
        EventModel.time <= finish,
        EventModel.page.in_(pages)
      )
      .group_by(
        bucket,
        EventModel.page
      )
      .order_by(
        bucket,
        EventModel.page
      )
  )
  compiled_query = query.compile(compile_kwargs={"literal_binds": True})
  print(compiled_query)
  results = session.exec(query).fetchall()
  pprint(results)

SELECT time_bucket('1 min'::interval, eventmodel.time) AS time_bucket_1, eventmodel.page, count(*) AS count_1 
FROM eventmodel 
WHERE eventmodel.time > '2026-04-19 08:01:50.634062+00:00' AND eventmodel.time <= '2026-04-19 10:01:50.634120+00:00' AND eventmodel.page IN ('/about', '/contact', '/pages', '/pricing') GROUP BY time_bucket('1 min'::interval, eventmodel.time), eventmodel.page ORDER BY time_bucket('1 min'::interval, eventmodel.time), eventmodel.page
[(datetime.datetime(2026, 4, 19, 9, 47, tzinfo=datetime.timezone.utc), '/about', 46),
 (datetime.datetime(2026, 4, 19, 9, 47, tzinfo=datetime.timezone.utc), '/contact', 55),
 (datetime.datetime(2026, 4, 19, 9, 47, tzinfo=datetime.timezone.utc), '/pages', 50),
 (datetime.datetime(2026, 4, 19, 9, 47, tzinfo=datetime.timezone.utc), '/pricing', 49)]
